# 02 · Cleaning & validation
Rules: timezone normalisation, de-duplication, physical bounds, flat-line removal, robust-z outlier *flags*, ≥2/4 slots per hour, ≤2 h gap interpolation only.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / "backend"))
import pandas as pd, numpy as np, json
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [2]:
q = pd.read_csv(ROOT/'reports/data_quality/quality_log.csv')
q.pivot_table(index='check', columns='action', values='count', aggfunc='sum').fillna(0).astype(int)

action,flagged (kept),hour set to NaN,linear interpolation (marked),removed
check,,,,
flatline_>=6h,0,0,0,124234
gap_<=2h_interpolated,0,0,6184,0
hour_with_<50%_slots,0,63836,0,0
out_of_physical_bounds,0,0,0,46975
pm25_gt_pm10,1244,0,0,0
robust_z>6.0,4651,0,0,0


In [3]:
sel = pd.read_csv(ROOT/'reports/data_quality/station_selection.csv'); sel

,location_id,pm25_coverage_pct,hours,selected_for_model
0,3409331,70.9,15175,True
1,3409435,76.0,15335,True
2,3409436,79.2,15335,True
3,3409437,85.4,15325,True
4,3409438,73.8,15310,True
5,3409439,47.7,15334,False
6,3409523,76.5,15335,True
7,3409526,90.5,15335,True
8,3409528,81.9,15334,True
9,3410005,61.8,15335,False


In [4]:
cov = pd.read_csv(ROOT/'reports/data_quality/monthly_pm25_coverage.csv')
cov.pivot(index='month', columns='location_id', values='pm25_coverage_pct').round(0)

location_id,3409331,3409435,3409436,3409437,3409438,3409439,3409523,3409526,3409528,3410005
month,,,,,,,,,,
2025-02,92.0,76.0,95.0,59.0,98.0,96.0,91.0,95.0,84.0,95.0
2025-03,92.0,95.0,94.0,72.0,82.0,93.0,86.0,85.0,64.0,38.0
2025-04,77.0,83.0,82.0,76.0,66.0,81.0,77.0,65.0,79.0,14.0
2025-05,84.0,88.0,84.0,83.0,84.0,82.0,71.0,80.0,60.0,23.0
2025-06,84.0,85.0,85.0,80.0,70.0,80.0,68.0,85.0,83.0,2.0
2025-07,84.0,86.0,89.0,94.0,84.0,63.0,92.0,89.0,88.0,52.0
2025-08,92.0,79.0,84.0,91.0,81.0,2.0,90.0,91.0,91.0,32.0
2025-09,51.0,94.0,89.0,91.0,85.0,2.0,92.0,92.0,90.0,60.0
2025-10,0.0,91.0,87.0,88.0,72.0,4.0,89.0,91.0,80.0,61.0


In [5]:
h = pd.read_parquet(ROOT/'data/processed/hourly_measurements.parquet')
print(h.shape, h.ts.min(), h.ts.max())
h[['pm25','pm10','no2','so2','co','o3']].describe().round(1)

(153153, 23) 2025-02-18 20:00:00+00:00 2026-11-19 18:00:00+00:00


,pm25,pm10,no2,so2,co,o3
count,113899.0,108441.0,105818.0,52496.0,105200.0,105953.0
mean,36.0,83.9,23.6,3.3,0.9,25.6
std,36.2,62.4,21.3,8.9,0.7,28.4
min,0.0,0.0,0.0,0.0,0.0,0.0
25%,14.9,42.9,10.0,1.3,0.4,7.6
50%,25.1,67.7,17.4,2.1,0.7,15.8
75%,46.7,108.3,29.2,3.2,1.1,34.6
max,1000.0,1654.1,565.6,429.3,10.0,772.0
